# CCA-F 考试复习 Notebook — Agentic Architecture

本 notebook 按你给定的 CCA-F task 列表重写。每个 Task 都包含概念解释、可运行 Python 示例、考点总结和 2 道模拟题。

## 环境初始化

In [ ]:
import json
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional

MODEL = "claude-haiku-4-5-20251001"
print("环境就绪：本 notebook 只使用本地模拟，不需要 API key；示例结构贴近 Anthropic Messages API / Claude Code 工作流。")

## D1.1 — 设计并实现自主任务执行的智能体循环

### 📖 概念解释

智能体循环是宿主程序围绕 Claude 工具使用建立的控制结构。模型响应中的 `stop_reason` 决定下一步：`tool_use` 表示模型请求外部工具，宿主程序必须执行所有 `tool_use` 块，并把带有匹配 `tool_use_id` 的 `tool_result` 作为下一条 `user` 消息追加回历史；`end_turn` 才表示模型本轮已经完成，可以把最终文本交给用户。生产实现还需要设置最大迭代次数作为防护，但它不是正常终止条件。

CCA-F 常考边界：不要用“任务完成”等自然语言判断终止，不要漏追加 assistant 的原始 `tool_use` 消息，不要只处理第一个工具调用，也不要把工具结果塞进错误角色。心智模型是：模型负责选择下一步动作，代码负责执行动作、维护消息顺序、保存状态并执行结构化停止判断。

In [ ]:
# Task 1.1：智能体循环本地模拟
# 重点：stop_reason == "tool_use" 继续；stop_reason == "end_turn" 结束。
# 本示例模拟 Claude 一次响应中可能请求多个工具。

def execute_tool(name: str, tool_input: dict) -> str:
    if name == "get_customer":
        return json.dumps({"customer_id": "CUST-1001", "verified": True}, ensure_ascii=False)
    if name == "lookup_order":
        return json.dumps({"order_id": tool_input["order_id"], "amount": 89.0, "eligible": True}, ensure_ascii=False)
    if name == "process_refund":
        return json.dumps({"refund_id": "RF-7788", "status": "submitted"}, ensure_ascii=False)
    return json.dumps({"error": f"unknown tool: {name}"}, ensure_ascii=False)


fake_model_turns = [
    {
        "stop_reason": "tool_use",
        "content": [
            {"type": "tool_use", "id": "toolu_customer", "name": "get_customer", "input": {"email": "a@example.com"}},
            {"type": "tool_use", "id": "toolu_order", "name": "lookup_order", "input": {"order_id": "ORD-9"}},
        ],
    },
    {
        "stop_reason": "tool_use",
        "content": [
            {"type": "tool_use", "id": "toolu_refund", "name": "process_refund", "input": {"order_id": "ORD-9", "amount": 89.0}}
        ],
    },
    {"stop_reason": "end_turn", "content": [{"type": "text", "text": "客户已验证，订单符合退款条件，退款已提交。"}]},
]

messages = [{"role": "user", "content": "帮我处理 ORD-9 的退款"}]
max_iterations = 5

for iteration, turn in enumerate(fake_model_turns, start=1):
    if iteration > max_iterations:
        raise RuntimeError("达到迭代上限：这是安全保护，不是正常完成条件")

    print("stop_reason:", turn["stop_reason"])
    if turn["stop_reason"] == "tool_use":
        # 先保留 assistant 的 tool_use，再追加 user 的 tool_result；顺序不能反。
        messages.append({"role": "assistant", "content": turn["content"]})
        tool_results = []
        for block in turn["content"]:
            result = execute_tool(block["name"], block["input"])
            tool_results.append({"type": "tool_result", "tool_use_id": block["id"], "content": result})
        messages.append({"role": "user", "content": tool_results})
    elif turn["stop_reason"] == "end_turn":
        print(turn["content"][0]["text"])
        break
    else:
        raise ValueError(f"未处理的 stop_reason: {turn['stop_reason']}")

print("消息历史长度:", len(messages))
print("最后一条工具结果:", messages[-1]["content"][0])

### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- 用 `stop_reason` 驱动循环：`tool_use` 继续，`end_turn` 结束。
- 对同一响应中的每个 `tool_use` 都执行工具，并返回匹配 `tool_use_id` 的 `tool_result`。
- 按 Anthropic 消息形态保留顺序：assistant `tool_use` 后接 user `tool_result`。
- 设置迭代上限、未知工具处理和异常路径，作为工程保护。

**✗ 反模式**

- 解析“完成了”“done”等文本作为终止条件。
- 只追加工具结果，漏掉 assistant 的 `tool_use` 消息。
- 只执行第一个工具调用，忽略同一响应里的其他工具。
- 把最大迭代次数当作正常业务终止条件。

### 🧪 模拟题

**Q1.** Claude 返回 `stop_reason: "tool_use"`，并且 `content` 中有两个 `tool_use` 块。宿主程序下一步应该做什么？

A) 执行两个工具，把各自带有匹配 `tool_use_id` 的 `tool_result` 追加到消息历史，再调用模型  
B) 只执行第一个工具，因为 Claude 每轮只能真正使用一个工具  
C) 直接把 assistant 文本返回给用户，等待用户确认  
D) 重新发送同一条 user 消息，直到模型返回 `end_turn`

> **答案：A**  
> **解析：** `tool_use` 是结构化工具请求；宿主程序必须处理同一响应中的所有工具调用，并用匹配 ID 回写结果。

**Q2.** 哪个消息顺序最符合 Anthropic 工具使用循环？

A) `user` 请求 → `assistant` 的 `tool_use` → `user` 的 `tool_result` → 下一次模型调用  
B) `user` 请求 → `user` 的 `tool_result` → `assistant` 的 `tool_use` → 下一次模型调用  
C) `user` 请求 → 直接执行工具 → 丢弃消息历史 → 下一次模型调用  
D) `assistant` 最终文本 → `tool_result` → `tool_use` → 结束

> **答案：A**  
> **解析：** 模型先发出 `tool_use`，宿主程序执行后以 user 角色追加对应 `tool_result`；顺序错误会破坏工具调用上下文。

## D1.2 — 使用协调者-子智能体模式编排多智能体系统

### 📖 概念解释

协调者-子智能体编排通常采用 Hub-and-Spoke：协调者是唯一的任务入口和质量控制中心，负责拆解任务、选择子智能体、传递显式上下文、隔离工具权限、汇总结果、检查冲突和覆盖缺口。子智能体专注于局部任务，不应默认共享会话历史，也不应形成难以追踪的点对点通信网络。

CCA-F 的判断重点不是“多智能体更高级”，而是何时值得拆分：任务范围宽、可并行、需要不同工具权限、上下文超长或需要独立验证时，子智能体有价值；简单线性任务使用单智能体或普通代码更可靠。考试干扰项常把协调者降级成路由器，或忽略聚合阶段的冲突处理。

In [ ]:
# Task 1.2：Hub-and-Spoke 多智能体编排模拟
# 协调者负责拆解、上下文传递、聚合和覆盖检查；子智能体之间不直接通信。

def subagent(role: str, assignment: str, context: dict) -> dict:
    simulated_findings = {
        "market_research": "需求集中在中小团队，证据来自访谈摘要。",
        "policy_review": "退款限制来自 refund-policy.md#auto。",
        "risk_review": "高金额退款需要人工审批。",
    }
    return {
        "role": role,
        "assignment": assignment,
        "received_context_keys": sorted(context.keys()),
        "finding": simulated_findings[role],
        "confidence": 0.82 if role != "risk_review" else 0.76,
    }


def coordinator(topic: str) -> dict:
    base_context = {
        "topic": topic,
        "required_output": "claim-source mapping",
        "source_policy": "优先使用一手来源",
    }
    assignments = {
        "market_research": "识别用户需求与证据来源",
        "policy_review": "检查政策条款与适用条件",
        "risk_review": "找出需要人工升级的风险",
    }
    required_roles = set(assignments)
    results = [subagent(role, task, base_context) for role, task in assignments.items()]
    returned_roles = {item["role"] for item in results}

    low_confidence = [item["role"] for item in results if item["confidence"] < 0.8]
    return {
        "topic": topic,
        "results": results,
        "coverage_gaps": sorted(required_roles - returned_roles),
        "needs_follow_up": low_confidence,
        "final_summary": "协调者汇总所有子任务，并标记低置信度领域。",
    }


print(json.dumps(coordinator("自动退款流程改造"), ensure_ascii=False, indent=2))

### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- 协调者维护覆盖地图，负责任务拆分、上下文路由、结果聚合、冲突检查和失败恢复。
- 子智能体接收最小但完整的上下文，只使用完成该子任务所需的工具。
- 对可并行、可隔离、可独立验证的任务使用子智能体；对简单线性任务保持简单。

**✗ 反模式**

- 所有请求都强行走多智能体流水线。
- 让子智能体彼此直接协调，导致状态和责任边界不可审计。
- 只拼接子智能体输出，不检查覆盖缺口、矛盾或低置信度结果。
- 拆分维度过窄，导致某些领域根本没有被分配。

### 🧪 模拟题

**Q1.** 研究系统漏掉音乐和电影，只覆盖视觉艺术，最可能的根因是什么？

A) 协调者任务分解过窄，没有设计覆盖地图  
B) 搜索子智能体没有足够 token，因此所有结论都无效  
C) 综合子智能体应该直接访问网页，而不是接收其他结果  
D) 应该让子智能体互相通信，自动发现遗漏领域

> **答案：A**  
> **解析：** 覆盖缺口通常来自协调者的拆分和覆盖地图设计，而不是单个子智能体能否更努力。

**Q2.** 哪种场景最适合使用协调者-子智能体模式？

A) 一个需要同时做政策审查、市场研究和风险评估的长任务，且各部分可独立验证  
B) 一个只需把摄氏温度转换为华氏温度的确定性计算  
C) 一个必须严格串行执行、每一步都依赖上一行代码输出的小脚本  
D) 一个没有外部工具、没有长上下文、没有质量检查要求的简单问答

> **答案：A**  
> **解析：** 多智能体适合宽范围、可并行、需隔离工具或独立验证的任务；简单确定性任务不应过度设计。

## D1.3 — 配置子智能体调用、上下文传递与生成

### 📖 概念解释

子智能体调用的关键是显式上下文和权限边界。Claude Code 中协调者通过 `Task` 工具生成子智能体；如果 `allowedTools` 没有包含 `Task`，提示词再强也不能创建子智能体。每次调用都应把目标、范围、已知事实、来源、输出格式、禁止事项和失败报告要求写入工作包，因为子智能体不会自动继承父会话历史。

还要区分“给子智能体上下文”和“泄露整个会话”。正确做法是传递完成任务所需的最小充分上下文，并为子智能体配置合适工具集。会话隔离让子智能体专注、可审计，也减少无关历史和敏感信息进入子任务。

In [ ]:
# Task 1.3：显式上下文注入、Task 权限与会话隔离模拟
# 子智能体不会自动继承父会话历史；协调者也必须被允许使用 Task。

def build_task_prompt(goal: str, known_facts: dict, output_schema: dict) -> str:
    return f"""目标：{goal}

范围：只判断自动退款资格；不要处理付款或发邮件。

已知事实（由协调者显式传入）：
{json.dumps(known_facts, ensure_ascii=False, indent=2)}

输出格式：
{json.dumps(output_schema, ensure_ascii=False, indent=2)}

失败报告：如果证据不足，返回 missing_context 字段并说明缺口。
"""


def invoke_subagent(coordinator_config: dict, child_allowed_tools: list[str], prompt: str) -> dict:
    if "Task" not in coordinator_config.get("allowedTools", []):
        raise PermissionError("协调者不能生成子智能体：allowedTools 缺少 Task")
    # 本地模拟：子智能体收到新的工作包，而不是父会话的完整历史。
    isolated_session = {"messages": [{"role": "user", "content": prompt}], "allowedTools": child_allowed_tools}
    return {
        "session_message_count": len(isolated_session["messages"]),
        "child_allowed_tools": isolated_session["allowedTools"],
        "inherits_parent_history": False,
    }


coordinator_config = {"allowedTools": ["Task", "Read", "Grep"]}
prompt = build_task_prompt(
    "分析 ORD-9 是否符合自动退款",
    {
        "customer_id": "CUST-1001",
        "order_id": "ORD-9",
        "policy_source": "refund-policy.md#auto",
        "order_amount": 89.0,
    },
    {"claim": "string", "evidence": "string", "source": "string", "confidence": "number", "missing_context": "list"},
)

print(prompt)
print(json.dumps(invoke_subagent(coordinator_config, ["Read", "Grep"], prompt), ensure_ascii=False, indent=2))

### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- 协调者的 `allowedTools` 必须包含 `Task`，才能生成子智能体。
- Task prompt 明确目标、范围、已知事实、来源、输出格式和失败报告要求。
- 子智能体工具权限按任务最小化配置，避免默认开放全部工具。
- 把子智能体视为隔离会话，而不是父对话的自动延续。

**✗ 反模式**

- 假设子智能体能读取父会话所有历史。
- 只写“继续刚才的研究”，没有传来源、证据和输出要求。
- 给协调者写了“请委派任务”的提示，但没有授予 `Task` 工具。
- 为方便而开放所有工具，破坏权限边界。

### 🧪 模拟题

**Q1.** 子智能体需要使用父会话中发现的来源，正确做法是什么？

A) 把目标、来源、摘录、已知事实和输出格式显式写入 Task prompt  
B) 假设子智能体自动读取父会话历史  
C) 只传一句“继续刚才的研究”  
D) 把所有工具都开放给子智能体，让它自己找上下文

> **答案：A**  
> **解析：** Task 调用是隔离工作包，必须显式传递完成任务所需的上下文。

**Q2.** 协调者提示词要求“生成一个研究子智能体”，但运行时无法生成。最可能的配置原因是什么？

A) 协调者的 `allowedTools` 中没有包含 `Task`  
B) 子智能体没有继承父会话标题  
C) 输出 schema 中缺少 `confidence` 字段  
D) 子智能体没有被命名为 researcher

> **答案：A**  
> **解析：** 子智能体生成依赖 `Task` 工具权限；提示词不能绕过工具权限配置。

## D1.4 — 实现 Hook 拦截机制与会话状态管理

### 📖 概念解释

Hook 是确定性控制点，不是提示词的同义词。`PreToolUse` 在工具执行前读取工具名、参数和会话状态，可以批准、修改或 veto 调用；`PostToolUse` 在工具结果进入模型上下文前进行规范化、审计记录或敏感字段处理。会话状态保存已经验证的客户、最近工具结果、风险标记和升级路径，确保高风险流程不能绕过前置步骤。

CCA-F 常把“模型承诺会遵守规则”和“程序化 enforcement”放在选项中对比。身份验证、退款上限、删除数据、外发邮件等顺序和权限要求，应该由 hook 和状态机强制执行；prompt 可以解释政策，但不能作为唯一控制层。

In [ ]:
# Task 1.4：Hook 拦截与会话状态管理
# 高风险操作必须靠程序化拦截，而不是只靠 system prompt。

@dataclass
class SupportSession:
    verified_customer_id: Optional[str] = None
    last_tool_results: list[dict] = field(default_factory=list)
    audit_log: list[dict] = field(default_factory=list)


def pre_tool_use(session: SupportSession, tool_name: str, tool_input: dict) -> dict:
    if tool_name in {"lookup_order", "process_refund"} and not session.verified_customer_id:
        return {"action": "veto", "reason": "must_verify_customer_first", "required_tool": "get_customer"}
    if tool_name == "process_refund" and tool_input.get("amount", 0) > 500:
        return {"action": "veto", "reason": "refund_above_auto_limit", "route": "human_approval"}
    return {"action": "allow"}


def post_tool_use(session: SupportSession, tool_name: str, result: dict) -> dict:
    normalized = dict(result)
    if "created_at" in normalized and isinstance(normalized["created_at"], int):
        normalized["created_at"] = datetime.fromtimestamp(normalized["created_at"]).isoformat()
    if "status_code" in normalized:
        normalized["status"] = {1: "active", 2: "suspended"}.get(normalized.pop("status_code"), "unknown")
    session.last_tool_results.append({"tool": tool_name, "result": normalized})
    session.audit_log.append({"tool": tool_name, "normalized": True})
    return normalized


def guarded_tool_call(session: SupportSession, tool_name: str, tool_input: dict, raw_result: dict) -> dict:
    decision = pre_tool_use(session, tool_name, tool_input)
    if decision["action"] == "veto":
        return {"blocked": True, **decision}
    return {"blocked": False, "result": post_tool_use(session, tool_name, raw_result)}


session = SupportSession()
print(guarded_tool_call(session, "process_refund", {"amount": 89}, {"refund_id": "RF-1"}))
session.verified_customer_id = "CUST-1001"
print(guarded_tool_call(session, "process_refund", {"amount": 89}, {"refund_id": "RF-7788", "status_code": 1, "created_at": 1705311000}))
print("审计日志:", session.audit_log)

### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- 用 `PreToolUse` 在执行前做顺序、权限和参数校验，必要时 veto。
- 用 `PostToolUse` 在结果进入模型上下文前做规范化、审计和脱敏。
- 在会话状态中记录已验证事实、工具结果和风险标记。
- 拦截时给出可恢复路径，例如要求先验证客户或升级人工审批。

**✗ 反模式**

- 只在 prompt 里说“退款前先验证”，没有程序化拦截。
- 让模型自行理解状态码、时间戳或内部错误格式。
- 拦截后只报错，不说明下一步应调用哪个工具或走哪条升级路径。
- 把 hook 当作普通日志，而不是执行前后的控制点。

### 🧪 模拟题

**Q1.** 退款前必须验证客户身份，最可靠的实现是什么？

A) 在 `PreToolUse` 中阻止 `process_refund`，直到会话中存在 `verified_customer_id`  
B) 在 system prompt 中多次强调退款前必须验证  
C) 让模型先输出“我确认已验证”再调用工具  
D) 把退款工具描述写得更详细

> **答案：A**  
> **解析：** 高风险顺序要求必须由程序化 hook 和会话状态强制执行。

**Q2.** 工具返回 `status_code: 1` 和 Unix 时间戳。希望模型看到稳定、可读、统一的结果，应该在哪里处理？

A) `PostToolUse`，在结果进入模型上下文前规范化字段  
B) `PreToolUse`，因为它发生在工具执行前  
C) 用户 prompt，因为用户最了解展示格式  
D) 不处理，让模型自行推断所有内部编码

> **答案：A**  
> **解析：** `PostToolUse` 适合对工具输出做规范化、脱敏和审计记录；`PreToolUse` 主要负责执行前拦截。